# SystemTopology `remove_component` Map Cleanup

Regression test for
[#625](https://github.com/sogno-platform/dpsim/issues/625):
`remove_component` must also drop the removed component from
`components_at_node`, otherwise the power flow solvers can still pick it
up while iterating that map during setup.

In [ ]:
import dpsimpy


def comp_names(system):
    return sorted(c.name() for c in system.components)


def map_names(system):
    return sorted(
        c.name() for comps in system.components_at_node.values() for c in comps
    )

In [ ]:
gnd = dpsimpy.emt.SimNode.gnd
n1 = dpsimpy.emt.SimNode("n1")
n2 = dpsimpy.emt.SimNode("n2")

vs = dpsimpy.emt.ph1.VoltageSource("vs")
vs.set_parameters(V_ref=complex(10, 0), f_src=50)
r1 = dpsimpy.emt.ph1.Resistor("r1")
r1.set_parameters(R=1)
c1 = dpsimpy.emt.ph1.Capacitor("c1")
c1.set_parameters(C=1e-3)

vs.connect([gnd, n1])
r1.connect([n1, n2])
c1.connect([n2, gnd])

system = dpsimpy.SystemTopology(50, [gnd, n1, n2], [vs, r1, c1])
assert "r1" in map_names(system)
print("before:", comp_names(system), "| map:", map_names(system))

In [ ]:
system.remove_component("r1")
print("after:", comp_names(system), "| map:", map_names(system))

assert comp_names(system) == ["c1", "vs"]
assert "r1" not in map_names(system)
assert "c1" in map_names(system) and "vs" in map_names(system)
print("removed component is gone from components_at_node: ok")

In [ ]:
system.remove_component("does_not_exist")
assert comp_names(system) == ["c1", "vs"]
print("removing a non-existent component is a no-op: ok")

In [ ]:
d1 = dpsimpy.emt.SimNode("d1")
d2 = dpsimpy.emt.SimNode("d2")
ra = dpsimpy.emt.ph1.Resistor("dup")
ra.set_parameters(R=1)
ra.connect([d1, d2])
rb = dpsimpy.emt.ph1.Resistor("dup")
rb.set_parameters(R=2)
rb.connect([d1, d2])

s2 = dpsimpy.SystemTopology(50, [d1, d2], [ra, rb])
s2.remove_component("dup")

assert comp_names(s2) == []
assert map_names(s2) == []
print("all same-named components purged from the map: ok")